In [1]:
from investment_agents.data.clients.finmind import FinMindClient

from investment_agents.data.repositories.price import PriceRepository
from investment_agents.data.repositories.dividend import DividendRepository
from investment_agents.data.repositories.institutional import InstitutionalRepository
from investment_agents.data.repositories.market_regime import MarketRegimeRepository

from investment_agents.features.price_adjustment import PriceAdjustmentService
from investment_agents.features.technical import TechnicalFeatureService
from investment_agents.features.chip import ChipFeatureService
from investment_agents.features.regime import RegimeFeatureService

from investment_agents.snapshots.technical import TechnicalSnapshotService
from investment_agents.snapshots.chip import ChipSnapshotService
from investment_agents.snapshots.regime import MarketRegimeSnapshotService

from investment_agents.agents.technical import TechnicalAgent
from investment_agents.agents.chip import ChipAgent
from investment_agents.agents.regime import MarketRegimeAgent
from investment_agents.agents.portfolio_manager import PortfolioManagerAgent

from investment_agents.ranking.daily import DailyRankingService

2026-09-23 15:11:19.863 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-23 15:11:19.931 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success


In [2]:
client = FinMindClient()

price_repository = PriceRepository(client)
dividend_repository = DividendRepository(client)
institutional_repository = InstitutionalRepository(client)
regime_repository = MarketRegimeRepository(client)

In [3]:
price_adjustment_service = PriceAdjustmentService()

technical_feature_service = TechnicalFeatureService()

chip_feature_service = ChipFeatureService()

regime_feature_service = RegimeFeatureService()

In [4]:
technical_snapshot_service = TechnicalSnapshotService(
    price_repo=price_repository,
    dividend_repo=dividend_repository,
    adjustment_service=price_adjustment_service,
    technical_service=technical_feature_service,
)

chip_snapshot_service = ChipSnapshotService(
    institutional_repo=institutional_repository,
    price_repo=price_repository,
    chip_service=chip_feature_service,
)

regime_snapshot_service = MarketRegimeSnapshotService(
    repository=regime_repository,
    feature_service=regime_feature_service,
)

In [5]:
technical_agent = TechnicalAgent()
chip_agent = ChipAgent()
regime_agent = MarketRegimeAgent()
pm_agent = PortfolioManagerAgent()

ranking_service = DailyRankingService(
    technical_snapshot_service=technical_snapshot_service,
    chip_snapshot_service=chip_snapshot_service,
    regime_snapshot_service=regime_snapshot_service,

    technical_agent=technical_agent,
    chip_agent=chip_agent,
    regime_agent=regime_agent,
    pm_agent=pm_agent,
)

In [7]:
test_universe = [
    "2330",
    "2454",
    "2317",
    "6005",
    "2881",
    "2883"
]

as_of_date = "2026-09-21"

ranking_df = ranking_service.run(
    tickers=test_universe,
    as_of_date=as_of_date,
)

ranking_df

2026-09-22 00:29:49.408 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-22 00:29:49.612 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 
2026-09-22 00:30:00.385 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-22 00:30:00.466 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockDividendResult, data_id: 2330
2026-09-22 00:30:00.549 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-22 00:30:00.633 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2330
2026-09-22 00:30:09.173 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2454
2026-09-22 00:30:09.262 | INFO     | FinMind.data.finmi

,as_of_date,ticker,technical_score,chip_score,regime,regime_score,regime_confidence,final_score,conviction,rank
0,2026-09-21,2454,85,65,risk_on,78,82,75,65,1
1,2026-09-21,2330,72,55,risk_on,78,82,70,65,2
2,2026-09-21,2881,72,55,risk_on,78,82,70,65,3
3,2026-09-21,2883,85,60,risk_on,78,82,70,65,4
4,2026-09-21,2317,55,50,risk_on,78,82,65,60,5
5,2026-09-21,6005,55,50,risk_on,78,82,65,60,6


In [9]:
import time

from investment_agents.llm.factory import create_llm

llm = create_llm()

start = time.time()

response = llm.invoke(
    "Reply with exactly the word OK."
)

print(response.content)
print(f"{time.time() - start:.2f} sec")

OK
3.71 sec


In [11]:
from pydantic import BaseModel
import time


class TestOutput(BaseModel):
    score: int
    reason: str


llm = create_llm().with_structured_output(TestOutput)

start = time.time()

result = llm.invoke(
    """
    Return a score from 0 to 100.
    The evidence is moderately positive.
    Give a concise reason.
    """
)

print(result)
print(f"{time.time() - start:.2f} sec")

score=65 reason='The evidence shows some positive aspects but lacks strong support or has some limitations.'
8.99 sec


In [12]:
import time

chip_data = chip_snapshot_service.get_snapshot(
    ticker="2317",
    as_of_date="2026-09-18",
)

start = time.time()

chip_report = chip_agent.analyze(chip_data)

print(chip_report)
print(f"{time.time() - start:.2f} sec")

2026-09-21 23:13:15.904 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-09-21 23:13:16.118 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2317


score=50 reason='Mixed and divergent signals across investor groups and time horizons. Foreign Investors show strong short-term selling (-14.95% 1-day flow) but weak medium-term distribution (-0.32% 20-day flow). Investment Trusts exhibit medium-term accumulation (0.75% 20-day flow, 13/20 buying days) but recent negative streak. Dealers show contradictory patterns (positive 1-day buying vs. negative 5-day/20-day flows). No group demonstrates consistently strong or persistent accumulation/distribution across all horizons, leading to neutral/mixed institutional evidence.'
22.26 sec


In [6]:
chip_2317 = chip_snapshot_service.get_snapshot(
    ticker="2317",
    as_of_date="2026-09-18",
)

chip_2317

2026-09-21 01:35:29.738 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-09-21 01:35:29.929 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2317


{'foreign_flow_ratio_1d': -14.949026872393723,
 'foreign_flow_ratio_5d': -1.3012731278793255,
 'foreign_flow_ratio_20d': -0.32449807306897616,
 'foreign_buy_days_20d': 7.0,
 'foreign_streak': -1.0,
 'trust_flow_ratio_1d': -1.2145784410490903,
 'trust_flow_ratio_5d': 1.1770034172373036,
 'trust_flow_ratio_20d': 0.7541243186715478,
 'trust_buy_days_20d': 13.0,
 'trust_streak': -1.0,
 'dealer_flow_ratio_1d': 1.9561139016287639,
 'dealer_flow_ratio_5d': -1.173553940418045,
 'dealer_flow_ratio_20d': -0.3599471607566616,
 'dealer_buy_days_20d': 11.0,
 'dealer_streak': 2.0}

In [7]:
chip_data = chip_snapshot_service.get_snapshot(
    ticker="2317",
    as_of_date="2026-09-18",
)

chip_report = chip_agent.analyze(chip_data)

print("score:", chip_report.score)
print("reason:", chip_report.reason)

2026-09-21 01:35:37.060 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-09-21 01:35:37.127 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2317


score: 40
reason: The institutional chip-flow indicators present a mixed and somewhat conflicting picture. Foreign Investors show strong immediate selling pressure with a 1-day flow ratio of -14.95%, and their medium-term indicators (5-day and 20-day flow ratios) also suggest net selling, albeit at a reduced magnitude. The Investment Trusts, however, show a divergence with a positive 5-day and 20-day flow ratio, indicating some medium-term accumulation, despite a negative 1-day flow ratio and a current negative streak. Dealers present further divergence with a positive 1-day flow ratio but negative 5-day and 20-day flow ratios, suggesting short-term buying but medium-term selling. The mixed signals across investor groups and time horizons, particularly the divergence between Foreign Investors and Investment Trusts, lead to a moderately negative score, reflecting the stronger immediate selling pressure from Foreign Investors but acknowledging the conflicting evidence from other groups.
